In [10]:
import triton
import triton.language as tl

@triton.jit
def fused_SiLU_residual(residual_ptr, x_ptr, y_ptr, n_elements, BLOCK_SIZE:tl.constexpr):
  pid=tl.program_id(0)

  offsets=pid*BLOCK_SIZE+tl.arange(0, BLOCK_SIZE)
  mask=offsets<n_elements

  residual=tl.load(residual_ptr+offsets, mask=mask)

  x=tl.load(x_ptr+offsets,mask=mask)

  silu=(x*tl.sigmoid(x))

  final_output=residual+silu

  tl.store(y_ptr+offsets, final_output, mask=mask)


In [11]:
import torch
def fused_silu_residual(residual: torch.Tensor, x: torch.Tensor):
    assert residual.is_cuda
    assert x.is_cuda
    assert residual.shape == x.shape
    assert residual.is_contiguous()
    assert x.is_contiguous()

    y = torch.empty_like(x)

    n_elements = x.numel()

    grid = lambda meta: (
        triton.cdiv(n_elements, meta["BLOCK_SIZE"]),
    )

    fused_SiLU_residual[grid](
        residual,
        x,
        y,
        n_elements,
        BLOCK_SIZE=1024,
    )

    return y

In [12]:
x = torch.randn(1_000_000, device="cuda")
residual = torch.randn_like(x)

y = fused_silu_residual(residual, x)